In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set interactive plotting
%matplotlib inline
mpl.rcParams['figure.dpi'] = 150

print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

## 1. Setup

In [ ]:
from src.config import PipelineConfig

cfg = PipelineConfig(
    condition_a_dir   = Path('data/condition_A'),
    condition_b_dir   = Path('data/condition_B'),
    condition_a_label = 'Control',
    condition_b_label = 'Treatment',
    output_dir        = Path('figures_output'),
    cache_dir         = Path('.cache'),
    n_top_genes       = 200,   # set <= panel size for Xenium
    leiden_resolution = 0.5,
    dge_method        = 'wilcoxon',
    figure_format     = 'pdf',
    dpi               = 300,
)
print('Config ready. Output dir:', cfg.output_dir.absolute())

## 2. Load data

In [ ]:
from src.xenium_loader import load_two_conditions

adata = load_two_conditions(
    dir_a   = cfg.condition_a_dir,
    dir_b   = cfg.condition_b_dir,
    label_a = cfg.condition_a_label,
    label_b = cfg.condition_b_label,
)
print(adata)
print('Conditions:', adata.obs['condition'].value_counts().to_dict())

## 3. QC

In [ ]:
from src.preprocessing import run_qc
import scanpy as sc

# Calculate QC metrics before filtering
sc.pp.calculate_qc_metrics(adata, expr_type='counts', log1p=False, inplace=True)

# Inspect distributions
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
adata.obs['total_counts'].hist(bins=60, ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_xlabel('Total counts'); axes[0].set_title('Counts per cell')
adata.obs['n_genes_by_counts'].hist(bins=60, ax=axes[1], color='darkorange', edgecolor='none')
axes[1].set_xlabel('N genes'); axes[1].set_title('Genes per cell')
plt.tight_layout()
plt.show()

# Apply QC filter
adata = run_qc(
    adata,
    min_counts         = cfg.min_counts,
    max_counts         = cfg.max_counts,
    min_genes          = cfg.min_genes,
    max_genes          = cfg.max_genes,
    min_cells_per_gene = cfg.min_cells_per_gene,
)
print('After QC:', adata)

## 4. Preprocessing: Normalise, PCA, Harmony, UMAP, Cluster

In [ ]:
from src.preprocessing import (
    normalise_and_select_hvg,
    run_pca,
    run_harmony,
    build_graph_and_umap,
    run_leiden,
)

adata = normalise_and_select_hvg(adata, target_sum=cfg.target_sum, n_top_genes=cfg.n_top_genes)
adata = run_pca(adata, n_pcs=cfg.n_pcs, random_state=cfg.random_state)

# Plot variance explained
var_ratio = adata.uns['pca']['variance_ratio']
plt.figure(figsize=(4, 2))
plt.plot(np.cumsum(var_ratio[:30]) * 100, 'o-', ms=3, color='#0072B2')
plt.xlabel('PC'); plt.ylabel('Cumulative variance (%)'); plt.title('Elbow plot')
plt.tight_layout(); plt.show()

In [ ]:
adata = run_harmony(adata, batch_key=cfg.harmony_key, max_iter=cfg.harmony_max_iter)
adata = build_graph_and_umap(adata, use_harmony=True, n_neighbors=cfg.n_neighbors)
adata = run_leiden(adata, resolution=cfg.leiden_resolution, key_added=cfg.cluster_key)

print(f'Clusters: {adata.obs[cfg.cluster_key].nunique()}')
print(adata.obs[cfg.cluster_key].value_counts())

In [ ]:
# Quick inline UMAP
from src.figures import apply_nature_style
apply_nature_style()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
umap = adata.obsm['X_umap']

from src.figures import CONDITION_COLOURS, WONG, _cluster_palette
conditions = adata.obs['condition'].cat.categories.tolist()
for i, cond in enumerate(conditions):
    m = adata.obs['condition'] == cond
    axes[0].scatter(umap[m, 0], umap[m, 1], s=0.5, alpha=0.4,
                    c=CONDITION_COLOURS.get(cond, WONG[i]), label=cond, rasterized=True)
axes[0].set_title('UMAP — condition'); axes[0].legend(markerscale=6, frameon=False, fontsize=7)

clusters = sorted(adata.obs[cfg.cluster_key].unique(),
                  key=lambda x: (0, int(x)) if str(x).isdigit() else (1, str(x)))
pal = _cluster_palette(len(clusters))
for ci, cl in enumerate(clusters):
    m = adata.obs[cfg.cluster_key] == cl
    axes[1].scatter(umap[m, 0], umap[m, 1], s=0.5, alpha=0.5, c=pal[ci], label=cl, rasterized=True)
axes[1].set_title('UMAP — cluster'); axes[1].legend(markerscale=5, frameon=False, fontsize=6, ncol=2)

for ax in axes:
    ax.set_aspect('equal')
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.tight_layout(); plt.show()

## 5. Cell type annotation

In [ ]:
from src.cell_type_annotation import annotate_cell_types, BRAIN_MARKERS

adata = annotate_cell_types(adata, method='marker_scoring', markers=BRAIN_MARKERS)
print(adata.obs['cell_type'].value_counts())

## 6. Global DGE

In [ ]:
from src.dge_analysis import run_dge

dge = run_dge(
    adata,
    method        = cfg.dge_method,
    condition_key = cfg.dge_group_key,
    condition_a   = cfg.condition_a_label,
    condition_b   = cfg.condition_b_label,
)

# run_dge() always returns canonical column names
lfc_col = 'log2fc'
p_col   = 'pval_adj'

n_sig = ((dge[p_col] < 0.05) & (dge[lfc_col].abs() > 0.5)).sum()
print(f'Significant DEGs: {n_sig}')
dge.head(10)

## 7. Spatial statistics

In [ ]:
from src.spatial_stats import morans_i_scan, spatial_coexpression, neighborhood_enrichment

if 'spatial' in adata.obsm:
    # Run Moran's I on HVGs
    hvg_genes = adata.var[adata.var['highly_variable']].index.tolist()[:50]
    morans_df = morans_i_scan(adata, genes=hvg_genes, n_neighbors=6)
    print('Top spatially variable genes:')
    print(morans_df.head(10)[['gene', 'morans_i', 'p_adj']])
else:
    print('No spatial coordinates; skipping Moran\'s I.')
    morans_df = None

In [ ]:
coexpr = None
if morans_df is not None and len(morans_df) >= 4:
    top_svg = morans_df.head(12)['gene'].tolist()
    top_svg = [g for g in top_svg if g in adata.var_names]
    coexpr = spatial_coexpression(adata, genes=top_svg, n_neighbors=6)
    print('Co-expression matrix shape:', coexpr.shape)

In [ ]:
nbr = None
ct_key = 'cell_type' if 'cell_type' in adata.obs else cfg.cluster_key
if 'spatial' in adata.obsm:
    nbr = neighborhood_enrichment(adata, cell_type_key=ct_key, n_permutations=200)
    print('Neighbourhood enrichment z-score matrix:')
    print(nbr['z_score'].round(2))

## 8. Per-cluster DGE

In [ ]:
from src.cluster_dge import run_cluster_dge

group_key = 'cell_type' if 'cell_type' in adata.obs else cfg.cluster_key

cluster_dge = run_cluster_dge(
    adata,
    group_key     = group_key,
    condition_key = cfg.dge_group_key,
    condition_a   = cfg.condition_a_label,
    condition_b   = cfg.condition_b_label,
    method        = cfg.dge_method,
    min_cells_per_condition = 10,
    output_dir    = cfg.output_dir,
)

if not cluster_dge.empty:
    summary = cluster_dge.groupby('group')['direction'].value_counts().unstack(fill_value=0)
    print(summary)

## 9. Export all 11 Nature-grade figures

In [ ]:
from src.pipeline import XeniumDGEPipeline
from src import figures_extended as fe

# Inject preprocessed adata and dge back into pipeline for Fig 1-8
pipe = XeniumDGEPipeline(cfg)
pipe.adata = adata
pipe.dge_results = dge
pipe.make_figures()  # Figs 1-8

# Fig 9: cell types
if 'cell_type' in adata.obs:
    fe.plot_cell_type_panel(adata, output_dir=cfg.output_dir, fmt=cfg.figure_format)

# Fig 10: spatial stats
if morans_df is not None:
    fe.plot_spatial_stats(
        morans_df=morans_df,
        coexpr_matrix=coexpr,
        neighborhood_result=nbr,
        output_dir=cfg.output_dir, fmt=cfg.figure_format,
    )

# Fig 11: cluster DGE
if not cluster_dge.empty:
    fe.plot_cluster_dge(cluster_dge=cluster_dge, adata=adata,
                        output_dir=cfg.output_dir, fmt=cfg.figure_format)

print('All figures exported to', cfg.output_dir.absolute())

In [ ]:
# List output files
for f in sorted(cfg.output_dir.glob('*.pdf')):
    print(f.name)